# Data Quality — Validação e Monitoramento da Qualidade dos Dados

Este notebook consolida as regras de qualidade identificadas durante a exploração e o processamento da camada Silver.

O objetivo é automatizar verificações de qualidade sobre os dados tratados, permitindo identificar inconsistências antes que os dados sejam utilizados pelas camadas analíticas.

As validações contemplam:

- completude dos dados;
- duplicidade de registros;
- unicidade das chaves;
- validade de valores categóricos;
- consistência entre campos relacionados;
- integridade entre datasets;
- geração de métricas e status de qualidade.

## 1. Configuração e leitura da Silver

As validações são executadas sobre a camada Silver, pois ela representa os dados tratados e padronizados disponibilizados para consumo pelas etapas posteriores da pipeline.

A camada Bronze permanece preservada como fonte bruta e não será modificada por este processo.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

In [0]:
# Define o caminho da camada Silver utilizada nas validações de qualidade.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

SILVER_PATH = f"s3://{BUCKET_NAME}/silver/"

In [0]:
# Define os datasets da camada Silver que serão submetidos às validações.

DATASETS = [
    "avaliacao_alfabetizacao_municipio",
    "avaliacao_alfabetizacao_uf",
    "avaliacao_alunos",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]

In [0]:
# Carrega os datasets Silver em formato Parquet.

dataframes_silver = {}

for nome in DATASETS:
    caminho = f"{SILVER_PATH}{nome}/"

    df = spark.read.parquet(caminho)

    dataframes_silver[nome] = df

    print(
        f"[OK] {nome}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

## 2. Regras automáticas de qualidade

Nesta etapa são executadas validações automatizadas sobre os datasets da camada Silver.

As regras foram definidas a partir dos problemas identificados durante a exploração e o tratamento dos dados, permitindo consolidar os principais controles de qualidade em uma única execução.

In [0]:
# Define as chaves utilizadas nas validações de unicidade e integridade.

CHAVES = {
    "avaliacao_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "avaliacao_alfabetizacao_uf": ["ano", "sigla_uf", "rede"],
    "avaliacao_alunos": ["ano", "id_aluno"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf", "rede"]
}

In [0]:
# Inicializa a estrutura que armazenará os resultados das regras de Data Quality.

resultados_qualidade = []

In [0]:
# Valida a existência de valores nulos nas colunas que compõem as chaves principais de cada dataset.

for nome, chaves in CHAVES.items():
    df = dataframes_silver[nome]

    condicao_nula = F.lit(False)

    for coluna in chaves:
        condicao_nula = condicao_nula | F.col(coluna).isNull()

    qtd_invalidos = df.filter(condicao_nula).count()

    resultados_qualidade.append({
        "dataset": nome,
        "regra": "CHAVE_NULA",
        "registros_invalidos": qtd_invalidos,
        "status": "PASS" if qtd_invalidos == 0 else "FAIL"
    })

In [0]:
# Valida a unicidade das chaves principais de cada dataset.

for nome, chaves in CHAVES.items():
    df = dataframes_silver[nome]

    qtd_duplicadas = (
        df
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    resultados_qualidade.append({
        "dataset": nome,
        "regra": "CHAVE_DUPLICADA",
        "registros_invalidos": qtd_duplicadas,
        "status": "PASS" if qtd_duplicadas == 0 else "FAIL"
    })

In [0]:
# Consolida os resultados das primeiras regras de qualidade.

df_resultados_qualidade = spark.createDataFrame(resultados_qualidade)

display(
    df_resultados_qualidade
    .orderBy("dataset", "regra")
)

## 3. Validação de domínio e consistência

Além das chaves, são validados os domínios esperados para campos categóricos e as relações de consistência entre atributos.

Essas regras permitem identificar valores fora dos padrões definidos e combinações incompatíveis com a estrutura dos dados.

In [0]:
# Valida os domínios dos campos binários da base de alunos.

df_alunos = dataframes_silver["avaliacao_alunos"]

DOMINIOS_ALUNOS = {
    "presenca": [0, 1],
    "preenchimento_caderno": [0, 1],
    "alfabetizado": [0, 1]
}

for coluna, valores_validos in DOMINIOS_ALUNOS.items():

    qtd_invalidos = (
        df_alunos
        .filter(
            F.col(coluna).isNotNull()
            & ~F.col(coluna).isin(valores_validos)
        )
        .count()
    )

    resultados_qualidade.append({
        "dataset": "avaliacao_alunos",
        "regra": f"DOMINIO_{coluna.upper()}",
        "registros_invalidos": qtd_invalidos,
        "status": "PASS" if qtd_invalidos == 0 else "FAIL"
    })

In [0]:
# Valida a consistência entre preenchimento do caderno, proficiência e peso do aluno.

qtd_inconsistencias_caderno = (
    df_alunos
    .filter(
        (F.col("preenchimento_caderno") == 0)
        & (
            F.col("proficiencia").isNotNull()
            | F.col("peso_aluno").isNotNull()
        )
    )
    .count()
)

resultados_qualidade.append({
    "dataset": "avaliacao_alunos",
    "regra": "CONSISTENCIA_CADERNO",
    "registros_invalidos": qtd_inconsistencias_caderno,
    "status": "PASS" if qtd_inconsistencias_caderno == 0 else "FAIL"
})

In [0]:
# Valida se avaliações com caderno preenchido possuem proficiência e peso do aluno disponíveis.

qtd_avaliacoes_sem_resultado = (
    df_alunos
    .filter(
        (F.col("preenchimento_caderno") == 1)
        & (
            F.col("proficiencia").isNull()
            | F.col("peso_aluno").isNull()
        )
    )
    .count()
)

resultados_qualidade.append({
    "dataset": "avaliacao_alunos",
    "regra": "RESULTADO_AVALIACAO_PREENCHIDA",
    "registros_invalidos": qtd_avaliacoes_sem_resultado,
    "status": "PASS" if qtd_avaliacoes_sem_resultado == 0 else "FAIL"
})

In [0]:
# Atualiza a tabela consolidada com as novas regras de qualidade.

df_resultados_qualidade = spark.createDataFrame(
    resultados_qualidade
)

display(
    df_resultados_qualidade
    .orderBy("dataset", "regra")
)

## 4. Completude e faixas numéricas

Nesta etapa são avaliados campos essenciais e indicadores numéricos da camada Silver.

As validações de completude consideram apenas atributos cuja ausência representa efetivamente um problema de qualidade. Valores nulos estruturais, decorrentes das características dos dados, não são classificados automaticamente como erro.

Também são verificadas faixas esperadas para indicadores percentuais.

In [0]:
# Valida se as taxas de alfabetização estão dentro da faixa percentual esperada.

for nome in [
    "avaliacao_alfabetizacao_municipio",
    "avaliacao_alfabetizacao_uf"
]:
    df = dataframes_silver[nome]

    qtd_invalidos = (
        df
        .filter(
            F.col("taxa_alfabetizacao").isNotNull()
            & (
                (F.col("taxa_alfabetizacao") < 0)
                | (F.col("taxa_alfabetizacao") > 100)
            )
        )
        .count()
    )

    resultados_qualidade.append({
        "dataset": nome,
        "regra": "FAIXA_TAXA_ALFABETIZACAO",
        "registros_invalidos": qtd_invalidos,
        "status": "PASS" if qtd_invalidos == 0 else "FAIL"
    })

In [0]:
# Valida a faixa percentual de participação quando o campo estiver disponível nas bases de metas.

for nome in [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]:
    df = dataframes_silver[nome]

    qtd_invalidos = (
        df
        .filter(
            F.col("percentual_participacao").isNotNull()
            & (
                (F.col("percentual_participacao") < 0)
                | (F.col("percentual_participacao") > 100)
            )
        )
        .count()
    )

    resultados_qualidade.append({
        "dataset": nome,
        "regra": "FAIXA_PERCENTUAL_PARTICIPACAO",
        "registros_invalidos": qtd_invalidos,
        "status": "PASS" if qtd_invalidos == 0 else "FAIL"
    })

In [0]:
# Valida se todas as metas de alfabetização estão entre 0 e 100.

for nome in [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]:
    df = dataframes_silver[nome]

    colunas_metas = [
        coluna
        for coluna in df.columns
        if coluna.startswith("meta_alfabetizacao_")
    ]

    for coluna in colunas_metas:
        qtd_invalidos = (
            df
            .filter(
                F.col(coluna).isNotNull()
                & (
                    (F.col(coluna) < 0)
                    | (F.col(coluna) > 100)
                )
            )
            .count()
        )

        resultados_qualidade.append({
            "dataset": nome,
            "regra": f"FAIXA_{coluna.upper()}",
            "registros_invalidos": qtd_invalidos,
            "status": "PASS" if qtd_invalidos == 0 else "FAIL"
        })

In [0]:
df_resultados_qualidade = spark.createDataFrame(
    resultados_qualidade
)

display(
    df_resultados_qualidade
    .orderBy("dataset", "regra")
)

## 5. Integridade entre datasets

Nesta etapa é verificada a correspondência entre as bases de avaliação e metas nas granularidades de município e UF.

Diferenças de cobertura identificadas anteriormente são registradas como alertas (`WARN`) em vez de falhas, pois representam ausência de correspondência entre fontes e não necessariamente inconsistências nos dados.

In [0]:
# Valida a cobertura entre as bases municipais de avaliação e metas.

df_avaliacao_municipio = dataframes_silver[
    "avaliacao_alfabetizacao_municipio"
]

df_meta_municipio = dataframes_silver[
    "meta_alfabetizacao_municipio"
]

chaves_avaliacao_municipio = (
    df_avaliacao_municipio
    .select("ano", "id_municipio")
    .distinct()
)

chaves_meta_municipio = (
    df_meta_municipio
    .select("ano", "id_municipio")
    .distinct()
)

avaliacao_sem_meta = (
    chaves_avaliacao_municipio
    .join(
        chaves_meta_municipio,
        ["ano", "id_municipio"],
        "left_anti"
    )
    .count()
)

meta_sem_avaliacao = (
    chaves_meta_municipio
    .join(
        chaves_avaliacao_municipio,
        ["ano", "id_municipio"],
        "left_anti"
    )
    .count()
)

resultados_qualidade.append({
    "dataset": "avaliacao_meta_municipio",
    "regra": "AVALIACAO_SEM_META",
    "registros_invalidos": avaliacao_sem_meta,
    "status": "PASS" if avaliacao_sem_meta == 0 else "WARN"
})

resultados_qualidade.append({
    "dataset": "avaliacao_meta_municipio",
    "regra": "META_SEM_AVALIACAO",
    "registros_invalidos": meta_sem_avaliacao,
    "status": "PASS" if meta_sem_avaliacao == 0 else "WARN"
})

In [0]:
# Valida a cobertura entre as bases estaduais de avaliação e metas.

df_avaliacao_uf = dataframes_silver[
    "avaliacao_alfabetizacao_uf"
]

df_meta_uf = dataframes_silver[
    "meta_alfabetizacao_uf"
]

chaves_avaliacao_uf = (
    df_avaliacao_uf
    .select("ano", "sigla_uf")
    .distinct()
)

chaves_meta_uf = (
    df_meta_uf
    .select("ano", "sigla_uf")
    .distinct()
)

avaliacao_sem_meta_uf = (
    chaves_avaliacao_uf
    .join(
        chaves_meta_uf,
        ["ano", "sigla_uf"],
        "left_anti"
    )
    .count()
)

meta_sem_avaliacao_uf = (
    chaves_meta_uf
    .join(
        chaves_avaliacao_uf,
        ["ano", "sigla_uf"],
        "left_anti"
    )
    .count()
)

resultados_qualidade.append({
    "dataset": "avaliacao_meta_uf",
    "regra": "AVALIACAO_SEM_META",
    "registros_invalidos": avaliacao_sem_meta_uf,
    "status": "PASS" if avaliacao_sem_meta_uf == 0 else "WARN"
})

resultados_qualidade.append({
    "dataset": "avaliacao_meta_uf",
    "regra": "META_SEM_AVALIACAO",
    "registros_invalidos": meta_sem_avaliacao_uf,
    "status": "PASS" if meta_sem_avaliacao_uf == 0 else "WARN"
})

In [0]:
# Atualiza a visão consolidada com as validações de integridade entre fontes.

df_resultados_qualidade = spark.createDataFrame(
    resultados_qualidade
)

display(
    df_resultados_qualidade
    .orderBy("status", "dataset", "regra")
)

## 6. Resumo executivo de Data Quality

Os resultados das validações são consolidados por status para facilitar o monitoramento da qualidade dos dados.

Os status utilizados são:

- `PASS`: regra atendida;
- `WARN`: condição que merece acompanhamento, mas não bloqueia a pipeline;
- `FAIL`: violação crítica que deve interromper o processamento.

In [0]:
# Consolida a quantidade de regras por status.

resumo_status = (
    df_resultados_qualidade
    .groupBy("status")
    .agg(
        F.count("*").alias("quantidade_regras"),
        F.sum("registros_invalidos").alias("registros_afetados")
    )
    .orderBy("status")
)

display(resumo_status)

In [0]:
# Consolida os resultados de qualidade por dataset.

resumo_dataset = (
    df_resultados_qualidade
    .groupBy("dataset")
    .agg(
        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).alias("regras_pass"),

        F.sum(
            F.when(F.col("status") == "WARN", 1).otherwise(0)
        ).alias("regras_warn"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).alias("regras_fail")
    )
    .withColumn(
        "status_geral",
        F.when(F.col("regras_fail") > 0, "FAIL")
         .when(F.col("regras_warn") > 0, "WARN")
         .otherwise("PASS")
    )
)

display(resumo_dataset.orderBy("dataset"))

In [0]:
# Interrompe a execução caso exista alguma regra crítica com status FAIL.

quantidade_fail = (
    df_resultados_qualidade
    .filter(F.col("status") == "FAIL")
    .count()
)

quantidade_warn = (
    df_resultados_qualidade
    .filter(F.col("status") == "WARN")
    .count()
)

if quantidade_fail > 0:
    raise Exception(
        f"Data Quality falhou: {quantidade_fail} regra(s) com status FAIL."
    )

print(
    f"[OK] Data Quality concluído sem falhas críticas. "
    f"Warnings identificados: {quantidade_warn}."
)

## 7. Governança de dados com Unity Catalog

O projeto utiliza o Unity Catalog do Databricks como camada de governança e organização dos ativos de dados.

A estrutura configurada no ambiente é:

- **Catalog:** `workspace`
- **Schema:** `literacy_pipeline`
- **Volume:** `landing`

O schema `literacy_pipeline` centraliza os ativos relacionados ao projeto, enquanto o volume `landing` fornece uma área governada pelo Unity Catalog para disponibilização de arquivos no ambiente Databricks.

O armazenamento definitivo das camadas Bronze, Silver e Gold permanece no Amazon S3, mantendo a separação entre armazenamento e processamento adotada na arquitetura.

In [0]:
# Valida o schema utilizado pelo projeto no Unity Catalog.

spark.sql("""
    SHOW SCHEMAS IN workspace
""").filter(
    F.col("databaseName") == "literacy_pipeline"
).display()

In [0]:
# Valida os volumes registrados no schema do projeto.

spark.sql("""
    SHOW VOLUMES IN workspace.literacy_pipeline
""").display()

### 7.1 Acesso governado ao Amazon S3

O acesso do Databricks ao Data Lake no Amazon S3 é realizado por meio de objetos de governança do Unity Catalog.

A configuração utiliza:

- **Storage Credential** para representar a credencial de acesso à AWS;
- **External Location** para autorizar o acesso ao bucket S3 do projeto;
- separação entre armazenamento no S3 e processamento no Databricks;
- controle de acesso centralizado pelo Unity Catalog, evitando uso direto de credenciais AWS nos notebooks.

Essa abordagem reduz o acoplamento entre código e credenciais e melhora a governança do ambiente.

In [0]:
# Lista as External Locations disponíveis no Unity Catalog para validar o acesso governado ao Data Lake no Amazon S3.

spark.sql("""
    SHOW EXTERNAL LOCATIONS
""").display()

In [0]:
# Lista as Storage Credentials configuradas no Unity Catalog para validar a credencial utilizada na integração com a AWS.

spark.sql("""
    SHOW STORAGE CREDENTIALS
""").display()

### 7.2 Evidência de governança do acesso ao Data Lake

A External Location `literacy-data-pipeline` está associada ao bucket S3 utilizado pelas camadas Bronze, Silver e Gold.

O acesso ao Data Lake é realizado por meio do Unity Catalog, evitando a exposição de credenciais diretamente nos notebooks e centralizando a autorização de acesso ao armazenamento externo.

### 7.3 Lineage e limitações do ambiente

O Unity Catalog permite centralizar recursos de governança, incluindo organização dos ativos, controle de acesso e rastreabilidade dos dados.

No ambiente Databricks Free Edition utilizado neste projeto, a visualização de lineage dos ativos implementados não está disponível na configuração atual. Por esse motivo, o lineage não é apresentado como evidência prática da solução.

A governança implementada é demonstrada pela organização do projeto no Unity Catalog e pelo acesso controlado ao Amazon S3 por meio de Storage Credential e External Location.